In [ ]:
# ==========================================
# KAGGLE CELL 1: Install Dependencies
# Run this in the first cell of your notebook
# ==========================================
# !pip install -q polars sentence-transformers scikit-learn

# ==========================================
# KAGGLE CELL 2: Stage 4 Execution Script
# ==========================================
import os
import polars as pl
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

def run_semantic_diversity(input_parquet_path, target_sample_size=1000000, batch_size=2048):
    # Output goes to the current notebook's working directory
    output_path = "/kaggle/working/wmt_stage4_diverse.parquet"
    
    if not os.path.exists(input_parquet_path):
        raise FileNotFoundError(f"Could not find input file at {input_parquet_path}. Did you copy the path correctly?")
        
    print(f"--- Loading Stage 3 Checkpoint: {input_parquet_path} ---")
    df = pl.read_parquet(input_parquet_path)
    total_rows = df.height
    print(f"Loaded {total_rows:,} clean sentence pairs.")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device.upper()}")
    if device == "cpu":
        print("WARNING: GPU not detected. Turn on T4 x2 in Kaggle settings!")
    
    print("--- Loading LaBSE Model for Diversity Mapping ---")
    model = SentenceTransformer("sentence-transformers/LaBSE", device=device)
    
    # Extract source sentences and apply the "Space Hack" to avoid URL parsing bugs
    source_texts = [" " + text for text in df["source"].to_list()]
    
    # Stratified Approach: Create 10,000 macro-clusters instead of 1,000,000 tiny ones
    n_clusters = 10000
    samples_per_cluster = target_sample_size // n_clusters
    print(f"Targeting {n_clusters:,} macro-clusters. Will extract top {samples_per_cluster} pairs from each...")
    
    # Initialize K-Means 
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=10000, random_state=42, n_init="auto")
    
    print("--- Step 1: Fitting MiniBatchKMeans (Streaming Embeddings) ---")
    embed_buffer = []
    buffer_count = 0
    
    for i in range(0, total_rows, batch_size):
        batch_text = source_texts[i : i + batch_size]
        with torch.no_grad():
            batch_embeds = model.encode(batch_text, convert_to_tensor=False, show_progress_bar=False)
        
        embed_buffer.append(batch_embeds)
        buffer_count += len(batch_embeds)
        
        # Scikit-Learn Rule: The first batch passed to partial_fit MUST be >= n_clusters
        # We accumulate embeddings in RAM until we hit 15,000, then fit and clear to save memory
        if buffer_count >= 15000 or (i + batch_size) >= total_rows:
            X_train = np.vstack(embed_buffer)
            kmeans.partial_fit(X_train)
            
            embed_buffer = [] # Reset buffer
            buffer_count = 0
            
            processed = min(i + batch_size, total_rows)
            print(f"Clustering Train Progress: {processed:,} / {total_rows:,} rows fit...")
            
    print("--- Step 2: Assigning Clusters ---")
    all_labels = []
    
    for i in range(0, total_rows, batch_size):
        batch_text = source_texts[i : i + batch_size]
        with torch.no_grad():
            batch_embeds = model.encode(batch_text, convert_to_tensor=False, show_progress_bar=False)
        
        preds = kmeans.predict(batch_embeds)
        all_labels.extend(preds.tolist())
        
        if (i + batch_size) % (batch_size * 20) == 0 or (i + batch_size) >= total_rows:
            processed = min(i + batch_size, total_rows)
            print(f"Cluster Assignment Progress: {processed:,} / {total_rows:,} rows tagged...")
            
    # Attach the cluster IDs back onto our dataframe
    df = df.with_columns(pl.Series("cluster_id", all_labels))
    
    print("--- Step 3: Extracting Core Diverse Exemplars ---")
    # Group by the macro-clusters and keep the top N from each to reach 1,000,000
    # The hasattr check ensures it works on both new and older versions of Kaggle's Polars
    if hasattr(df, "group_by"):
        final_df = df.group_by("cluster_id").head(samples_per_cluster).drop("cluster_id")
    else:
        final_df = df.groupby("cluster_id").head(samples_per_cluster).drop("cluster_id")
    
    final_count = final_df.height
    print(f"\nFinal diverse row count after Stage 4: {final_count:,}")
    print(f"Redundant semantic duplicates dropped: {total_rows - final_count:,}")
    
    print(f"\n--- Saving Diversity Checkpoint to {output_path} ---")
    final_df.write_parquet(output_path)
    print("Stage 4 completed successfully! You can now use this output for Stage 5.")

# ==========================================
# EXECUTION: PASTE YOUR FILE PATH HERE
# ==========================================
INPUT_FILE_PATH = "/kaggle/input/notebooks/havishbalaga/notebookb511b59609/wmt_wmt19_zh_en_stage3.parquet" 

run_semantic_diversity(
    input_parquet_path=INPUT_FILE_PATH, 
    target_sample_size=1000000,   # Set to 1 million
    batch_size=2048
)